# DMPSP â€” Training on Kaggle (P100 16GB)

Trains all three DMPSP components on Kaggle's free P100 GPU (30h/week).

**Timeout-safe**: checkpoints every 2500 steps. Re-run any cell after timeout â€” `--resume` continues automatically.

**One-time Kaggle setup**:
1. Enable GPU: Settings â†’ Accelerator â†’ GPU P100
2. Enable internet: Settings â†’ Internet â†’ On
3. Run Cell 1 (installs deps) and Cell 2 (clones repo)
4. Optionally upload `data/processed/` as a Kaggle Dataset named `dmpsp-processed-data` to skip download

**Training order** (run sequentially â€” one GPU):
- Phase 1 â€” Action Proposal: ~3-4h (batch=64, 100K steps)
- Phase 2 â€” World Model: ~8-10h (batch=16, 100K steps, ReactionT5 fine-tuning)
- Phase 3 â€” Value Function: ~2-3h (batch=64, 100K steps)

**Encoder**: MorganFPEncoder (RDKit, no PyG needed). Created once during Phase 1, shared across all phases.

In [ ]:
# â”€â”€ CELL 1: Install dependencies â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Run once per session (~3-5 min). Kaggle has CUDA 12.1.
import subprocess, sys

pkgs = [
    # PyTorch + CUDA
    ('torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121', True),
    # PyG
    ('torch-geometric==2.4.0 torch-scatter==2.1.2 torch-sparse==0.6.18'
     ' -f https://data.pyg.org/whl/torch-2.4.0+cu121.html', True),
    # Rest
    ('transformers==4.44.0 accelerate==0.33.0 sentencepiece==0.2.0'
     ' rdkit==2024.03.5 httpx==0.27.2 PyYAML==6.0.2'
     ' pandas==2.2.2 pyarrow==17.0.0', False),
]

for pkg_str, use_index in pkgs:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + pkg_str.split()
    subprocess.run(cmd, check=True)

print('âœ“ Dependencies installed')

In [ ]:
# ── CELL 2: Clone repo ────────────────────────────────────────────────────
import os, subprocess
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/dmpsp')
_GIT_ENV = {
    **os.environ,
    'GIT_TERMINAL_PROMPT': '0',  # never prompt for credentials
    'GIT_ASKPASS': 'echo',       # return empty if asked for password
}

if not PROJECT_ROOT.exists():
    result = subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/thepavantejz/Diffusion-Model-Predictive-Synthesis-Planning.git',
         str(PROJECT_ROOT)],
        env=_GIT_ENV, capture_output=True, text=True,
    )
    if result.returncode != 0:
        print('CLONE FAILED:', result.stderr)
        raise RuntimeError(
            'git clone failed. Make the repo public on GitHub, or upload '
            'source as Kaggle dataset dmpsp-src and mount it.'
        )
    print(f'Cloned to {PROJECT_ROOT}')
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], env=_GIT_ENV, check=True)
    print(f'Repo up to date at {PROJECT_ROOT}')

for f in ['configs/model.yaml', 'dmpsp/encoder.py', 'train/train_proposal.py']:
    assert (PROJECT_ROOT / f).exists(), f'Missing: {f}'
print('Repo structure verified')

In [ ]:
# â”€â”€ CELL 3: Paths and GPU check â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os, sys, torch
from pathlib import Path

PROJECT_ROOT  = Path('/kaggle/working/dmpsp')
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
LOG_DIR        = Path('/kaggle/working/logs')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Data: uploaded dataset takes priority, else auto-download
UPLOADED = Path('/kaggle/input/dmpsp-processed-data')
DATA_DIR  = UPLOADED if (UPLOADED / 'trajectories_train.pkl').exists() \
            else Path('/kaggle/working/data/processed')

sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT   : {PROJECT_ROOT}')
print(f'DATA_DIR       : {DATA_DIR}  (exists={DATA_DIR.exists()})')
print(f'CHECKPOINT_DIR : {CHECKPOINT_DIR}')
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU            : {p.name}  ({p.total_memory/1e9:.1f} GB)')

In [ ]:
# â”€â”€ CELL 4: Prepare data (skip if already present) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Downloads USPTO-50K from HuggingFace and preprocesses to trajectories.
# Safe to re-run â€” skips if trajectories_train.pkl already exists.
import subprocess, sys
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/dmpsp')
UPLOADED     = Path('/kaggle/input/dmpsp-processed-data')
DATA_DIR     = UPLOADED if (UPLOADED / 'trajectories_train.pkl').exists() \
               else Path('/kaggle/working/data/processed')

if (DATA_DIR / 'trajectories_train.pkl').exists():
    print(f'âœ“ Data already at {DATA_DIR}')
else:
    print('Downloading + preprocessing USPTO-50K (~10 min) ...')
    r = subprocess.run(
        [sys.executable, str(PROJECT_ROOT / 'scripts' / 'prepare_data.py'),
         '--source', 'uspto50k',
         '--data_config', str(PROJECT_ROOT / 'configs' / 'data.yaml'),
         '--out_dir', str(DATA_DIR)],
    )
    if r.returncode != 0:
        raise RuntimeError('prepare_data.py failed')

import pickle
n_train = len(pickle.load(open(DATA_DIR / 'trajectories_train.pkl', 'rb')))
n_val   = len(pickle.load(open(DATA_DIR / 'trajectories_val.pkl',   'rb')))
print(f'âœ“ Train: {n_train:,}  Val: {n_val:,}')

---
## Phase 1 â€” Action Proposal Diffusion

Also creates `checkpoints/encoder/` on first run â€” the Morgan FP projection that
all three models share. Re-run after timeout: `--resume` continues from latest checkpoint.

In [ ]:
# â”€â”€ CELL 5: Train Action Proposal (100K steps, batch=64) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Re-run this cell to resume after session timeout.
import subprocess, sys
from pathlib import Path

PROJECT_ROOT   = Path('/kaggle/working/dmpsp')
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
UPLOADED = Path('/kaggle/input/dmpsp-processed-data')
DATA_DIR = UPLOADED if (UPLOADED / 'trajectories_train.pkl').exists() \
           else Path('/kaggle/working/data/processed')

r = subprocess.run([
    sys.executable,
    str(PROJECT_ROOT / 'train' / 'train_proposal.py'),
    '--model_config', str(PROJECT_ROOT / 'configs' / 'model.yaml'),
    '--train_config', str(PROJECT_ROOT / 'configs' / 'train.yaml'),
    '--data_dir',     str(DATA_DIR),
    '--out_dir',      str(CHECKPOINT_DIR / 'action_proposal'),
    '--encoder_dir',  str(CHECKPOINT_DIR / 'encoder'),
    '--device',       'cuda',
    '--batch_size',   '64',
    '--max_steps',    '100000',
    '--save_every',   '2500',
    '--log_level',    'INFO',
    '--resume',
])
print(f'Exit: {r.returncode}')

## Phase 2 â€” World Model (ReactionT5 fine-tuning)

Heaviest step. Loads the shared encoder from `checkpoints/encoder/` (created in Phase 1).
Gradient checkpointing enabled. ~8-10h total; safe to split across sessions.

In [ ]:
# â”€â”€ CELL 6: Train World Model (100K steps, batch=16) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import subprocess, sys
from pathlib import Path

PROJECT_ROOT   = Path('/kaggle/working/dmpsp')
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
UPLOADED = Path('/kaggle/input/dmpsp-processed-data')
DATA_DIR = UPLOADED if (UPLOADED / 'trajectories_train.pkl').exists() \
           else Path('/kaggle/working/data/processed')

r = subprocess.run([
    sys.executable,
    str(PROJECT_ROOT / 'train' / 'train_world_model.py'),
    '--model_config', str(PROJECT_ROOT / 'configs' / 'model.yaml'),
    '--train_config', str(PROJECT_ROOT / 'configs' / 'train.yaml'),
    '--data_dir',     str(DATA_DIR),
    '--out_dir',      str(CHECKPOINT_DIR / 'world_model'),
    '--encoder_dir',  str(CHECKPOINT_DIR / 'encoder'),
    '--device',       'cuda',
    '--batch_size',   '16',
    '--max_steps',    '100000',
    '--save_every',   '2500',
    '--log_level',    'INFO',
    '--resume',
])
print(f'Exit: {r.returncode}')

## Phase 3 â€” Value Function

In [ ]:
# â”€â”€ CELL 7: Train Value Function (100K steps, batch=64) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import subprocess, sys
from pathlib import Path

PROJECT_ROOT   = Path('/kaggle/working/dmpsp')
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
UPLOADED = Path('/kaggle/input/dmpsp-processed-data')
DATA_DIR = UPLOADED if (UPLOADED / 'trajectories_train.pkl').exists() \
           else Path('/kaggle/working/data/processed')

r = subprocess.run([
    sys.executable,
    str(PROJECT_ROOT / 'train' / 'train_value.py'),
    '--model_config', str(PROJECT_ROOT / 'configs' / 'model.yaml'),
    '--train_config', str(PROJECT_ROOT / 'configs' / 'train.yaml'),
    '--data_dir',     str(DATA_DIR),
    '--out_dir',      str(CHECKPOINT_DIR / 'value_fn'),
    '--encoder_dir',  str(CHECKPOINT_DIR / 'encoder'),
    '--device',       'cuda',
    '--batch_size',   '64',
    '--max_steps',    '100000',
    '--save_every',   '2500',
    '--log_level',    'INFO',
    '--resume',
])
print(f'Exit: {r.returncode}')

In [ ]:
# â”€â”€ CELL 8: Checkpoint status â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from pathlib import Path

CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')

for name in ['encoder', 'action_proposal', 'world_model', 'value_fn']:
    d = CHECKPOINT_DIR / name
    if not d.exists():
        print(f'  {name:<20} NOT STARTED')
        continue
    ckpts = sorted(d.glob('checkpoint_step*.pt'))
    if not ckpts:
        print(f'  {name:<20} directory exists, no checkpoints yet')
    else:
        latest = ckpts[-1]
        step = int(latest.stem.replace('checkpoint_step', ''))
        target = 1 if name == 'encoder' else 100000
        pct = step / target * 100
        mb = latest.stat().st_size / 1e6
        print(f'  {name:<20} step {step:>6,} / {target:>6,}  ({pct:5.1f}%)  {mb:.0f} MB  [{len(ckpts)} ckpts]')

In [ ]:
# â”€â”€ CELL 9: Smoke test â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import json, subprocess, sys
from pathlib import Path

PROJECT_ROOT   = Path('/kaggle/working/dmpsp')
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')

r = subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'plan_route.py'),
        '--smiles',         'CC(=O)Oc1ccccc1C(=O)O',
        '--checkpoint_dir', str(CHECKPOINT_DIR),
        '--device',         'cuda',
        '--max_steps',      '3',
        '--log_level',      'WARNING',
    ],
    capture_output=True, text=True,
)

if r.returncode == 0:
    route = json.loads(r.stdout)
    print(f"Steps            : {route['n_steps']}")
    print(f"Yield            : {route['total_yield_fraction']:.3f}")
    print(f"Planning time    : {route['planning_time_seconds']:.2f}s")
    print(f"Objective scores : {route['objective_scores']}")
else:
    print('FAILED:', r.stderr[-500:])

In [ ]:
# â”€â”€ CELL 10: Evaluate pharma benchmark â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import subprocess, sys
from pathlib import Path

PROJECT_ROOT   = Path('/kaggle/working/dmpsp')
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')

subprocess.run([
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'evaluate.py'),
    '--checkpoint_dir', str(CHECKPOINT_DIR),
    '--device',         'cuda',
    '--max_steps',      '5',
    '--out_json',       '/kaggle/working/results/evaluation.json',
])

In [ ]:
# â”€â”€ CELL 11: Benchmark (beam vs MCTS vs random, 100 test molecules) â”€â”€â”€â”€â”€â”€â”€
import subprocess, sys
from pathlib import Path

PROJECT_ROOT   = Path('/kaggle/working/dmpsp')
CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')
UPLOADED = Path('/kaggle/input/dmpsp-processed-data')
DATA_DIR = UPLOADED if (UPLOADED / 'trajectories_train.pkl').exists() \
           else Path('/kaggle/working/data/processed')

subprocess.run([
    sys.executable,
    str(PROJECT_ROOT / 'scripts' / 'benchmark.py'),
    '--checkpoint_dir', str(CHECKPOINT_DIR),
    '--data_dir',       str(DATA_DIR),
    '--device',         'cuda',
    '--n_molecules',    '100',
    '--max_steps',      '5',
    '--out_csv',        '/kaggle/working/results/benchmark.csv',
    '--out_json',       '/kaggle/working/results/benchmark.json',
])

In [ ]:
# â”€â”€ CELL 12: Zip and download checkpoints â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Kaggle keeps /kaggle/working/ output after session ends.
# Download from: notebook â†’ Output tab â†’ Files.
import shutil
from pathlib import Path

CHECKPOINT_DIR = Path('/kaggle/working/checkpoints')

shutil.make_archive('/kaggle/working/dmpsp_checkpoints', 'zip', str(CHECKPOINT_DIR))
size_mb = Path('/kaggle/working/dmpsp_checkpoints.zip').stat().st_size / 1e6
print(f'âœ“ /kaggle/working/dmpsp_checkpoints.zip  ({size_mb:.0f} MB)')
print('Download: Output tab â†’ Files â†’ dmpsp_checkpoints.zip')